# CSCI 4253 / 5253 - Lab #4 - Patent Problem with Spark RDD - SOLUTION
<div>
 <h2> CSCI 4283 / 5253 
  <IMG SRC="https://www.colorado.edu/cs/profiles/express/themes/cuspirit/logo.png" WIDTH=50 ALIGN="right"/> </h2>
</div>

This [Spark cheatsheet](https://s3.amazonaws.com/assets.datacamp.com/blog_assets/PySpark_SQL_Cheat_Sheet_Python.pdf) is useful

In [1]:
from pyspark import SparkContext, SparkConf
import numpy as np
import operator

In [2]:
conf=SparkConf().setAppName("Lab4-rdd").setMaster("local[*]")
sc = SparkContext(conf=conf)

Using PySpark and RDD's on the https://coding.csel.io machines is slow -- most of the code is executed in Python and this is much less efficient than the java-based code using the PySpark dataframes. Be patient and trying using `.cache()` to cache the output of joins. You may want to start with a reduced set of data before running the full task. You can use the `sample()` method to extract just a sample of the data or use 

These two RDD's are called "rawCitations" and "rawPatents" because you probably want to process them futher (e.g. convert them to integer types, etc). 

The `textFile` function returns data in strings. This should work fine for this lab.

Other methods you use might return data in type `Byte`. If you haven't used Python `Byte` types before, google it. You can convert a value of `x` type byte into e.g. a UTF8 string using `x.decode('uft-8')`. Alternatively, you can use the `open` method of the gzip library to read in all the lines as UTF-8 strings like this:
```
import gzip
with gzip.open('cite75_99.txt.gz', 'rt',encoding='utf-8') as f:
    rddCitations = sc.parallelize( f.readlines() )
```
This is less efficient than using `textFile` because `textFile` would use the underlying HDFS or other file system to read the file across all the worker nodes while the using `gzip.open()...readlines()` will read all the data in the frontend and then distribute it to all the worker nodes.

In [3]:
rddCitations = sc.textFile("cite75_99.txt.gz")
rddPatents = sc.textFile("apat63_99.txt.gz")

The data looks like the following.

In [4]:
rddCitations.take(5)

['"CITING","CITED"',
 '3858241,956203',
 '3858241,1324234',
 '3858241,3398406',
 '3858241,3557384']

In [5]:
rddPatents.take(5)

['"PATENT","GYEAR","GDATE","APPYEAR","COUNTRY","POSTATE","ASSIGNEE","ASSCODE","CLAIMS","NCLASS","CAT","SUBCAT","CMADE","CRECEIVE","RATIOCIT","GENERAL","ORIGINAL","FWDAPLAG","BCKGTLAG","SELFCTUB","SELFCTLB","SECDUPBD","SECDLWBD"',
 '3070801,1963,1096,,"BE","",,1,,269,6,69,,1,,0,,,,,,,',
 '3070802,1963,1096,,"US","TX",,1,,2,6,63,,0,,,,,,,,,',
 '3070803,1963,1096,,"US","IL",,1,,2,6,63,,9,,0.3704,,,,,,,',
 '3070804,1963,1096,,"US","OH",,1,,2,6,63,,3,,0.6667,,,,,,,']

In other words, they are a single string with multiple CSV's. You will need to convert these to (K,V) pairs, probably convert the keys to `int` and so on. You'll need to `filter` out the header string as well since there's no easy way to extract all the lines except the first.

### Step 1: Parse both files into (key, value) pairs
The header lines start with a quote, so they are filtered out. Each remaining line is split on commas. Citations become `(citing, cited)` pairs and patents become `(patent number, list of the other fields)`. Both are pair RDDs, so they can be joined. The row counts, 16,522,438 citations and 2,923,922 patents, match the DataFrame notebook.

In [6]:
%%time
cites = (rddCitations
         .filter(lambda line: not line.startswith('"'))
         .map(lambda line: line.split(","))
         .map(lambda f: (int(f[0]), int(f[1]))))

pats = (rddPatents
        .filter(lambda line: not line.startswith('"'))
        .map(lambda line: line.split(","))
        .map(lambda f: (int(f[0]), f[1:])))

print("cores:", sc.defaultParallelism)
print(cites.take(3))
print(pats.take(2))
print("rows:", cites.count(), pats.count())

cores: 2
[(3858241, 956203), (3858241, 1324234), (3858241, 3398406)]
[(3070801, ['1963', '1096', '', '"BE"', '""', '', '1', '', '269', '6', '69', '', '1', '', '0', '', '', '', '', '', '', '']), (3070802, ['1963', '1096', '', '"US"', '"TX"', '', '1', '', '2', '6', '63', '', '0', '', '', '', '', '', '', '', '', ''])]
rows: 16522438 2923922
CPU times: user 25.4 ms, sys: 4.87 ms, total: 30.3 ms
Wall time: 35.2 s


### Step 2: Build a (patent, state) lookup
The state is field 4 of each patent's field list. Its quote characters are stripped, and patents with no state (stored as `""`) are dropped, which is the RDD version of filtering out NULLs. 1,784,989 patents have a state, the same as the DataFrame notebook. The lookup is cached because both joins use it.

In [7]:
%%time
state = (pats
         .map(lambda kv: (kv[0], kv[1][4].strip('"')))
         .filter(lambda kv: kv[1] != "")
         .cache())

print(state.take(4))
print("patents with a state:", state.count())

[(3070802, 'TX'), (3070803, 'IL'), (3070804, 'OH'), (3070805, 'CA')]
patents with a state: 1784989
CPU times: user 5.9 ms, sys: 8.67 ms, total: 14.6 ms
Wall time: 8.27 s


### Step 3: Test the pipeline on a sample
To debug cheaply, the sample is every citation made by patents 5,950,000 to 5,959,999 (109,253 rows). Each citing patent keeps its whole citation list, so its count is exact. The pipeline re-keys the citations by the cited patent and joins with the state lookup, re-keys by the citing patent and joins again, keeps the pairs where the two states are equal, and counts them per citing patent with `reduceByKey`. The sample returned 125, 98, 96 and 90 for patents 5959466, 5952345, 5958954 and 5951547, the same counts as the DataFrame solution.

In [8]:
%%time
sampleCites = cites.filter(lambda c: 5950000 <= c[0] < 5960000).cache()
print("sample citation rows:", sampleCites.count())

byCited  = sampleCites.map(lambda c: (c[1], c[0]))      # (cited, citing)
j1       = byCited.join(state)                          # (cited, (citing, cited_state))
byCiting = j1.map(lambda x: (x[1][0], x[1][1]))         # (citing, cited_state)
j2       = byCiting.join(state)                         # (citing, (cited_state, citing_state))
same     = j2.filter(lambda x: x[1][0] == x[1][1])      # keep same-state pairs only
counts   = same.map(lambda x: (x[0], 1)).reduceByKey(operator.add)   # (citing, count)

print(counts.takeOrdered(10, key=lambda kv: (-kv[1], kv[0])))

sample citation rows: 109253
[(5959466, 125), (5952345, 98), (5958954, 96), (5951547, 90), (5957893, 68), (5958794, 48), (5951243, 43), (5955751, 40), (5957906, 38), (5957898, 37)]
CPU times: user 42.9 ms, sys: 9.62 ms, total: 52.6 ms
Wall time: 1min 29s


### Step 4: Run on the full data
The same pipeline runs on all 16.5 million citations, using 16 partitions for each shuffle so no worker has to hold too much data at once. The result is cached. 571,919 patents have at least one same-state citation, the same as the DataFrame solution, and the top 10 counts are identical. This took about 4 minutes on 2 cores.

In [9]:
%%time
byCited  = cites.map(lambda c: (c[1], c[0]))             # (cited, citing)
j1       = byCited.join(state, 16)                       # (cited, (citing, cited_state))
byCiting = j1.map(lambda x: (x[1][0], x[1][1]))          # (citing, cited_state)
j2       = byCiting.join(state, 16)                      # (citing, (cited_state, citing_state))
same     = j2.filter(lambda x: x[1][0] == x[1][1])       # keep same-state pairs only
counts   = same.map(lambda x: (x[0], 1)).reduceByKey(operator.add, 16).cache()

print("patents with a same-state citation:", counts.count())
print(counts.takeOrdered(10, key=lambda kv: (-kv[1], kv[0])))

patents with a same-state citation: 571919
[(5959466, 125), (5983822, 103), (6008204, 100), (5952345, 98), (5958954, 96), (5998655, 96), (5936426, 94), (5739256, 90), (5913855, 90), (5925042, 90)]
CPU times: user 60.9 ms, sys: 19.9 ms, total: 80.7 ms
Wall time: 4min 13s


### Step 5: Add the count to each patent and show the top 10
Joining the patent rows with the counts appends the same-state count to the end of each patent's field list. Only patents with at least one same-state citation appear, because this is an inner join. The top 10 is sorted by count, highest first, with patent number as the tie-break so the order is fixed.

In [10]:
%%time
newPatents = (pats.join(counts, 16)
                  .map(lambda kv: (kv[0], kv[1][0] + [kv[1][1]])))   # (patent, [fields..., same_state_count])

for row in newPatents.takeOrdered(10, key=lambda kv: (-kv[1][-1], kv[0])):
    print(row)

(5959466, ['1999', '14515', '1997', '"US"', '"CA"', '5310', '2', '', '326', '4', '46', '159', '0', '1', '', '0.6186', '', '4.8868', '0.0455', '0.044', '', '', 125])
(5983822, ['1999', '14564', '1998', '"US"', '"TX"', '569900', '2', '', '114', '5', '55', '200', '0', '0.995', '', '0.7201', '', '12.45', '0', '0', '', '', 103])
(6008204, ['1999', '14606', '1998', '"US"', '"CA"', '749584', '2', '', '514', '3', '31', '121', '0', '1', '', '0.7415', '', '5', '0.0085', '0.0083', '', '', 100])
(5952345, ['1999', '14501', '1997', '"US"', '"CA"', '749584', '2', '', '514', '3', '31', '118', '0', '1', '', '0.7442', '', '5.1102', '0', '0', '', '', 98])
(5958954, ['1999', '14515', '1997', '"US"', '"CA"', '749584', '2', '', '514', '3', '31', '116', '0', '1', '', '0.7397', '', '5.181', '0', '0', '', '', 96])
(5998655, ['1999', '14585', '1998', '"US"', '"CA"', '', '1', '', '560', '1', '14', '114', '0', '1', '', '0.7387', '', '5.1667', '', '', '', '', 96])
(5936426, ['1999', '14466', '1997', '"US"', '"CA"

### Result
The top 10 counts are 125, 103, 100, 98, 96, 96, 94, 90, 90, 90, the same as the DataFrame solution and the reference output. As in the DataFrame notebook, six patents tie at 90, so which three appear in the top 10 depends on the tie-break. Here ties are ordered by patent number.